# Exploratary Data Analysis

- Problem Statement: The goal of this project is to train a CNN model to predict what the Pancam image contains based on the image passed into it to aid researchers in picking what images they want to look at for example they want to see soil data the model would then pick only the images that contain soil

## Imports

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.preprocessing import MultiLabelBinarizer
import cv2


## Data

In [2]:
# Read in text files with the filenames and labels 
def read_txt_files(path):

    rows = []

    with open(path, encoding='UTF-8') as f:
        
        for line in f:

            line = line.strip()

            filename = line.split(",")[0]

            labels = line.split(",")[1:]

            rows.append({'filename':filename, 'labels':labels})

        data = pd.DataFrame(rows)

        return data

test_data = read_txt_files('rover_image_classification/mer-labeled-data-set-v1.1.0/test-set-v1.1.0.txt')

train_data = read_txt_files('rover_image_classification/mer-labeled-data-set-v1.1.0/train-set-v1.1.0.txt')

val_data = read_txt_files('rover_image_classification/mer-labeled-data-set-v1.1.0/val-set-v1.1.0.txt')


In [3]:
# We have in 55986 train data image note that this includes the augmented data aswell
print('Train Data Shape: ', train_data.shape)

# 742 Test data image
print('Test Data Shape: ', test_data.shape)

# 14136 val data images
print('Val Data Shape', val_data.shape)


Train Data Shape:  (55986, 2)
Test Data Shape:  (742, 2)
Val Data Shape (14136, 2)


In [4]:
# Around 30 rows in the train data are not in the images so will drop these rows doesnt seem to be the same in val and test data so will leave as is there
image_dir = Path('rover_image_classification/mer-labeled-data-set-v1.1.0/images')

folder_files = []

for f in image_dir.iterdir():

    folder_files.append(f.name)

folder_files = pd.DataFrame(folder_files,columns=['filename'])

# No missing files in these datasets
print('Missing Files in Val Data: ', len(val_data[~val_data['filename'].isin(folder_files['filename'])]))
print('Missing Files in Test Data: ', len(test_data[~test_data['filename'].isin(folder_files['filename'])]))

# 30 missing files in Train
print('Missing Files in Train Data Data: ', len(train_data[~train_data['filename'].isin(folder_files['filename'])]))

drop_train_data = train_data[~train_data['filename'].isin(folder_files['filename'])].index

train_data.drop(drop_train_data,inplace=True)

Missing Files in Val Data:  0
Missing Files in Test Data:  0
Missing Files in Train Data Data:  30


In [27]:
# Filtering the data to only contain the original images of the dataset
orig_train_data = train_data[train_data['filename'].str.endswith('img.jpg')].copy()
orig_test_data = test_data[test_data['filename'].str.endswith('img.jpg')].copy()
orig_val_data = val_data[val_data['filename'].str.endswith('img.jpg')].copy()

orig_train_data['split'] = 'train'
orig_test_data['split'] = 'test'
orig_val_data['split'] = 'val'

# Getting the properties of each image
def get_properties(images):

    with Image.open(image_dir/images) as im:

        image_props = {'width':im.width, 'height':im.height, 'mode':im.mode}

        array = np.asarray(im.convert('L'), dtype='float32')

        image_props['mean_intensity'] = array.mean()
        image_props['std_intensity'] = array.std()
        image_props['min_intensity'] = array.min()
        image_props['max_intensity'] = array.max()
        hist = np.bincount(array.astype(np.uint8).ravel(), minlength=256) / array.size
        hist = hist[hist >0]
        image_props['entropy'] = -(hist * np.log2(hist)).sum()
        image_props['sharpness'] = cv2.Laplacian(array, cv2.CV_32F).var()
        image_props['pct_black'] = (array <= 2).mean() * 100
        image_props['pct_white'] = (array >= 253).mean() * 100
        image_props['noise'] = (array - cv2.GaussianBlur(array, (5,5), 0)).std()

    return pd.Series(image_props)


orig_train_data_properties = orig_train_data['filename'].apply(get_properties)
orig_test_data_properties = orig_test_data['filename'].apply(get_properties)
orig_val_data_properties = orig_val_data['filename'].apply(get_properties)

orig_train_data = orig_train_data.join(orig_train_data_properties)
orig_test_data = orig_test_data.join(orig_test_data_properties)
orig_val_data = orig_val_data.join(orig_val_data_properties)

# Concatenating all data into one df to perform analysis on all data
data = pd.concat([orig_train_data, orig_val_data, orig_test_data], ignore_index=True)

data.to_csv('im_stats.csv' ,index=False)

In [28]:
data = pd.read_csv('im_stats.csv')

In [29]:
data['split'].value_counts()
# Taking only the original iamges we have, 1805 train images, 742 test and 456 val 

split
train    1805
test      742
val       456
Name: count, dtype: int64

In [30]:
mlb = MultiLabelBinarizer()
y = pd.DataFrame(mlb.fit_transform(data['labels']), columns=mlb.classes_, index=data['filename'])

### File Quality

In [31]:
data.isnull().sum()
# No null values 

filename          0
labels            0
split             0
width             0
height            0
mode              0
mean_intensity    0
std_intensity     0
min_intensity     0
max_intensity     0
entropy           0
sharpness         0
pct_black         0
pct_white         0
noise             0
dtype: int64

In [32]:
data['filename'].duplicated().sum()
# No duplicates

np.int64(0)

In [33]:
aug_val = val_data[~val_data['filename'].str.endswith('img.jpg')]
aug_test = test_data[~test_data['filename'].str.endswith('img.jpg')]
aug_train = train_data[~train_data['filename'].str.endswith('img.jpg')]

print('No. of Augmented val images: ', len(aug_val))
print('No. of Augmented test images: ', len(aug_test))
print('No. of Augmented train images: ', len(aug_train))

# No augmented test images
# 1368 augmented val images
# 54151 augmented train images

No. of Augmented val images:  13680
No. of Augmented test images:  0
No. of Augmented train images:  54151


## Image Statistics

### Per Image Features 